weaviateライブラリーをインストール

In [ ]:
!pip install -U weaviate-client

ライブラリーをインポート

In [ ]:
import base64
import dotenv
import glob
import IPython
import json
import os

import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Property, DataType, Configure, Multi2VecField
from weaviate.classes.query import MetadataQuery


print("weaviate:", weaviate.__version__)

In [ ]:
dotenv.load_dotenv()
GOOGLE_AI_STUDIO_API_KEY = os.getenv("GOOGLE_AI_STUDIO_API_KEY")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
WEAVIATE_URL = os.getenv("WEAVIATE_URL")
GOOGLE_AI_STUDIO_PROJECT_ID = os.getenv("GOOGLE_AI_STUDIO_PROJECT_ID")

基本的には<br> https://docs.weaviate.io/weaviate/model-providers/google/embeddings-multimodal <br>の通りでOKだけれど、google AI studioのAPIを使う場合のトラップは2つ<br>1. vector_config=Configure.Vectors.multi2vec_googleではなくてvector_config=Configure.Vectors.multi2vec_google_gemini<br>2. vector_config=Configure.Vectors.multi2vec_google_geminiで引数modelでgemini-embedding-2-previewを指定する

weaviateへ接続

In [ ]:
header = {"X-Goog-Studio-Api-Key": GOOGLE_AI_STUDIO_API_KEY}
# https://weaviate-python-client.readthedocs.io/en/latest/weaviate.html#weaviate.connect_to_weaviate_cloud
client = weaviate.connect_to_weaviate_cloud(cluster_url=WEAVIATE_URL,
                                            auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
                                            headers=header)

In [ ]:
client.is_ready()

weaviateにcollection(DBの大枠の箱みたいなもの)を作成する

In [ ]:
# https://docs.weaviate.io/weaviate/model-providers/google/embeddings-multimodal#vectorizer-parameters
# https://docs.weaviate.io/weaviate/config-refs/datatypes
client.collections.create(name="my_collection",
                          properties=[Property(name="name",
                                               data_type=DataType.TEXT),
                                      Property(name="description",
                                               data_type=DataType.TEXT),
                                      Property(name="tags",
                                               data_type=DataType.TEXT_ARRAY),  # TEXT_ARRAYの場合はtext_fieldsのembedding対象に出来ない
                                      Property(name="image",
                                               data_type=DataType.BLOB),
                                      Property(name="image_url",
                                               data_type=DataType.TEXT)],
                          vector_config=Configure.Vectors.multi2vec_google_gemini(image_fields=[Multi2VecField(name="image",
                                                                                                               weight=0.9)],  # embeddingでimage(画像)は重みを0.9にする
                                                                                  text_fields=[Multi2VecField(name="description",
                                                                                                              weight=0.1)],  # embeddingでdescription(説明文)は重みを0.1にする
                                                                                  model="gemini-embedding-2-preview"))

In [ ]:
# print(dir(Configure.Vectors))

weaviateのcollectionにデータを保存する

In [ ]:
def read_json_file(json_file_path):
    with open(file=json_file_path, mode="r", encoding="utf-8") as f:
        json_to_dict = json.load(f)
    return json_to_dict

In [ ]:
def get_jpeg_file_path_list(directory_path):
    jpeg_file_path_list = glob.glob(pathname="{a}/*.jpg".format(a=directory_path))
    return jpeg_file_path_list

In [ ]:
# https://docs.weaviate.io/weaviate/search/image#create-a-base64-representation-of-an-online-image
def image_to_base64(img_path):
    with open(file=img_path, mode="rb") as f:
        image_binary = f.read()
    image_base64 = base64.b64encode(s=image_binary).decode("utf-8")
    return image_base64

In [ ]:
product_json_path = "/hogehoge/for_weaviate/product_information.json"
product_info_dict = read_json_file(json_file_path=product_json_path)
product_info_list = product_info_dict["products"]
product_image_dir_path = "/hogehoge/for_weaviate"
product_image_path_list = get_jpeg_file_path_list(directory_path=product_image_dir_path)

In [ ]:
source_object_list = []
for i in range(len(product_info_list)):
    source_object = {}
    source_object["name"] = product_info_list[i]["name"]
    source_object["description"] = product_info_list[i]["description"]
    source_object["tags"] = product_info_list[i]["tags"]
    source_object["image_base64"] = image_to_base64(img_path=product_image_path_list[i])
    source_object["image_url"] = product_info_list[i]["image_url"]
    source_object_list.append(source_object)

In [ ]:
collection = client.collections.get("my_collection")

In [ ]:
# https://docs.weaviate.io/weaviate/model-providers/google/embeddings-multimodal#data-import
with collection.batch.fixed_size(batch_size=200) as batch:
    for source_object in source_object_list:
        weaviate_object = {"name": source_object["name"],
                           "description": source_object["description"],
                           "tags": source_object["tags"],
                           "image": source_object["image_base64"],
                           "image_url": source_object["image_url"]}
        batch.add_object(properties=weaviate_object)

In [ ]:
# import inspect


# print(inspect.signature(Configure.Vectors.multi2vec_google_gemini))

検索してみる

In [ ]:
collection = client.collections.get("my_collection")

In [ ]:
# https://docs.weaviate.io/weaviate/search/hybrid
response = collection.query.hybrid(query="夏でも快適な部屋着でお勧めを教えて。",
                                   alpha=0.75,  # ベクトル検索とキーワード検索の割合 # 設定値はベクトル検索の割合値
                                   return_metadata=MetadataQuery(score=True,
                                                                 explain_score=True),
                                   limit=5)

In [ ]:
for i in range(len(response.objects)):
    print(response.objects[i].properties["name"])
    print(response.objects[i].properties["description"])
    print(response.objects[i].properties["tags"])
    print(response.objects[i].metadata.score)
    print(response.objects[i].metadata.explain_score)

In [ ]:
IPython.display.Image(url=response.objects[0].properties["image_url"])

In [ ]:
# for res_obj in response.objects:
#     print(res_obj.properties["name"])
#     IPython.display.Image(url=res_obj.properties["image_url"])

FastAPIへのAPIテスト

In [ ]:
import requests

In [ ]:
response = requests.post(url="http://192.168.10.1:10002/search_product_database_by_weaviate_hybrid_search",
                         headers={"Content-Type": "application/json"},
                         json={"query": "夏でも快適な部屋着でお勧めを教えて。"})

In [ ]:
print(response.json())

weaviateとの接続を切断

In [ ]:
client.close()

In [ ]:
print(client.is_ready())

collectionを削除する場合

In [ ]:
client.collections.delete("my_collection")